# Named Entity Recognition

In [ ]:
import spacy
from nltk import sent_tokenize

## Load model

In [ ]:
def load_model():
    nlp_model = spacy.load("en_core_web_trf")
    return nlp_model

In [ ]:
nlp_model = load_model()

## Load dataset

In [ ]:
import pandas as pd 
from glob import glob
def load_subtitles_dataset(dataset_path):
    substitles_paths = sorted(glob(dataset_path+"/*.ass"))
    scripts = []
    episode_num = []
    for path in substitles_paths:
        
        # Read Lines
        with open(path,'r') as file:
            lines = file.readlines()
            lines = lines[27:]
            rows = [",".join(line.split(',')[9:]) for line in lines]
        
        # Clean Output
        rows = [line.replace("\\N",' ') for line in rows]
        script = " ".join(rows)
        
        # Get Episode Number
        episode = int(path.split('-')[1].split('.')[0].strip())
        
        scripts.append(script)
        episode_num.append(episode)
    
    df = pd.DataFrame.from_dict({'episode':episode_num,'script':scripts})
    return df

In [ ]:
dataset_path = "../data/Subtitles/"
df= load_subtitles_dataset(dataset_path)

In [ ]:
sample_script = df.iloc[0]['script']

In [ ]:
sample_script

In [ ]:
sample_script.split('\n')[60:90]

In [ ]:
sentence = ".".join(sample_script.split('\n')[60:90])
sentence

## Run Model

In [ ]:
doc = nlp_model(sentence)

In [ ]:
doc.ents

In [ ]:
doc.ents[0].label_,doc.ents[1].label_,doc.ents[2].label_, doc.ents[-3].label_

In [ ]:
def get_ners_inference(script):
    script_sentences = sent_tokenize(script)

    ner_output = []
    
    for sentence in script_sentences:
        doc = nlp_model(sentence)
        ners = set()
        for ent in doc.ents: 
            if ent.label_=='PERSON':
                full_name = ent.text
                first_name=full_name.split(' ')[0]
                ners.add(first_name)
        ner_output.append(list(ners))
    return ner_output

In [ ]:
df = df.head(10)

In [ ]:
df['ners'] = df['script'].apply(get_ners_inference)

In [ ]:
df

# Chracter Network

In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import matplotlib.pyplot as plt

In [ ]:
def  generate_character_network(ner_df):
    
    df = ner_df
    
    window=10
    entity_relationship = []

    for row in df['ners']:
        previous_entities_in_window = []
        
        for sentence in row:
            previous_entities_in_window.append(sentence)
            previous_entities_in_window = previous_entities_in_window[-window:]
            
            previous_entities_flattened= sum(previous_entities_in_window, [])
            
            for entity in sentence:            
                for entity_in_window in previous_entities_flattened:
                    if entity!=entity_in_window:
                        entity_rel = sorted([entity,entity_in_window])
                        entity_relationship.append(entity_rel)
    
    relationship_df = pd.DataFrame({'value':entity_relationship})
    relationship_df['source'] = relationship_df['value'].apply(lambda x: x[0])
    relationship_df['target'] = relationship_df['value'].apply(lambda x: x[1])
    relationship_df = relationship_df.groupby(['source','target']).count().reset_index()
    relationship_df = relationship_df.sort_values('value',ascending=False)
    
    return relationship_df

In [ ]:
relationship_df = generate_character_network(df)

In [ ]:
relationship_df = relationship_df.sort_values('value',ascending=False)
relationship_df = relationship_df.head(200)

G = nx.from_pandas_edgelist(relationship_df, 
                            source = "source", 
                            target = "target", 
                            edge_attr = "value", 
                            create_using = nx.Graph())

net = Network(notebook = True, width="1000px", height="700px", bgcolor='#222222', font_color='white',cdn_resources='remote')
node_degree = dict(G.degree)

#Setting up node size attribute
nx.set_node_attributes(G, node_degree, 'size')
net.from_nx(G)
net.show("naruto.html")